In [113]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import PolynomialFeatures, FunctionTransformer,StandardScaler
import seaborn as sns
from sklearn.metrics import confusion_matrix,classification_report
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [114]:
df = pd.read_csv('pid-5M.csv')

In [ ]:
df.head()

,id,p,theta,beta,nphe,ein,eout
0,211,0.780041,1.081480,0.989962,0,0.000000,0.000000
1,211,0.260929,0.778892,0.902450,0,0.000000,0.000000
2,2212,0.773022,0.185953,0.642428,4,0.101900,0.000000
3,211,0.476997,0.445561,0.951471,0,0.000000,0.000000
4,2212,2.123290,0.337332,0.908652,0,0.034379,0.049256


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000000 entries, 0 to 4999999
Data columns (total 7 columns):
 #   Column  Dtype  
---  ------  -----  
 0   id      int64  
 1   p       float64
 2   theta   float64
 3   beta    float64
 4   nphe    int64  
 5   ein     float64
 6   eout    float64
dtypes: float64(5), int64(2)
memory usage: 267.0 MB


In [ ]:
sns.pairplot(df.sample(30000))

In [115]:
df['id'] = np.where(df['id'] == -11, 'positron',
                    np.where(df['id'] == 211, 'pion',
                             np.where(df['id'] == 321, 'kaon', 'proton'))
                    )

In [117]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns='id'),df['id'] , random_state=42, test_size=0.2)

In [133]:
custom_weights = {
    'kaon': 4.5,
    'pion': 0.8,
    'positron': 50.0,
    'proton': 0.64
}

In [135]:
skewed_energy_cols = [4, 5]
high_degree_cols = [0, 2]
standard_cols = [0,1,2,3,4,5]

energy_pipeline = Pipeline([
    ('log', FunctionTransformer(np.log1p, validate=True))
])

physics_curve_pipeline = Pipeline([
    ('poly_high', PolynomialFeatures(degree=4, interaction_only=False, include_bias=False))
])

standard_pipeline = Pipeline([
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(transformers=[
    ('energy_log', energy_pipeline, skewed_energy_cols),
    ('physics_poly', physics_curve_pipeline, high_degree_cols),
    ('standard_scale', standard_pipeline, standard_cols)
], remainder='drop')


final_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        class_weight=custom_weights,
        solver='lbfgs',
        max_iter=500,
        tol=1e-3,
        random_state=42,
        n_jobs=-1
    ))
])

In [136]:
final_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('energy_log',
                                                  Pipeline(steps=[('log',
                                                                   FunctionTransformer(func=<ufunc 'log1p'>,
                                                                                       validate=True))]),
                                                  [4, 5]),
                                                 ('physics_poly',
                                                  Pipeline(steps=[('poly_high',
                                                                   PolynomialFeatures(degree=4,
                                                                                      include_bias=False))]),
                                                  [0, 2]),
                                                 ('standard_scale',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  [0, 1, 2, 3, 4, 5])])),
                ('classifier',
                 LogisticRegression(class_weight={'kaon': 4.5, 'pion': 0.8,
                                                  'positron': 50.0,
                                                  'proton': 0.64},
                                    max_iter=500, n_jobs=-1, random_state=42,
                                    tol=0.001))])

In [137]:
y_pred = final_pipeline.predict(X_test)

In [138]:
accuracy_score(y_test, y_pred)

0.929331

In [139]:
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[ 42304   2846    225    998]
 [ 35100 506675  15300   4781]
 [     2    454   2570      0]
 [ 10640    291     32 377782]]
              precision    recall  f1-score   support

        kaon       0.48      0.91      0.63     46373
        pion       0.99      0.90      0.95    561856
    positron       0.14      0.85      0.24      3026
      proton       0.98      0.97      0.98    388745

    accuracy                           0.93   1000000
   macro avg       0.65      0.91      0.70   1000000
weighted avg       0.96      0.93      0.94   1000000

